## Requirement 
#### I have a Client and the client having one of Source schema <br> ( employees and HR) in PDF,i want build a AI Data Model <br>to automation of best Data Model Creation <br> and Its Loading and Mapping Document and <br> SnowFlake SQL Queries based on requirement Submitted

## Solution Design - RAG

```mermaid
graph TD
    A[Read PDF schema]
    B[Extract text]
    C[Split text into chunks]
    D[Generate embeddings]
    E[Store embeddings in Chroma DB]
    F[User query]
    G[Retrieve relevant chunks]
    H[Build LLM prompt with context]
    I[LLM generates answer]
    J[Return final output/HTML]

    A --> B
    B --> C
    C --> D
    D --> E
    F --> G
    E --> G
    G --> H
    H --> I
    I --> J

### Load Environment Variables

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is available.")
else:
    print("OPENAI_API_KEY is not available.") 

OPENAI_API_KEY is available.


### Making Open AI LLM Initialization

In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-nano", temperature=0) 

### Extracing Text From Data Model PDF

In [5]:
from langchain_community.document_loaders import PyPDFLoader
admDocloader=PyPDFLoader("./Source/NN_ADM_Schema.pdf")
schmadocs=admDocloader.load()

In [6]:
schmadocs

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-07-05T18:55:24+05:30', 'author': 'Naresh Neelam', 'moddate': '2026-07-05T18:55:24+05:30', 'source': './Source/NN_ADM_Schema.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='4.3 NN Sample Schema Table Descriptions \n \nConsider the columns of each table of HR sample schema. \n• Table NN.ADM_COUNTRIES \n• Table NN.ADM_DEPARTMENTS \n• Table NN.ADM_EMPLOYEES \n• Table NN.ADM_JOBS \n• Table NN.ADM_JOB_HISTORY \n• Table NN.ADM_LOCATIONS \n• Table NN.ADM_REGIONS \n \n43.1 Table NN.ADM_COUNTRIES \nTable NN.ADM_COUNTRIES Table Description \nColumn Name Null? Type \nCOUNTRY_ID NOT NULL CHAR(2) \nCOUNTRY_NAME   VARCHAR2(40) \nREGION_ID   NUMBER \n4.3.2 Table NN.ADM_DEPARTMENTS \nTable  NN.ADM_DEPARTMENTS Table Description \nColumn Name Null? Type \nDEPARTMENT_ID NOT NULL NUMBER(4) \nDEPARTMENT_NAME NOT NULL VARCHAR2(30) \nMANAGER_ID   NUMBER(6) \nLOCATION_ID   NUMBE

### Create Own Metadata for PDF Chunks

In [7]:
for i in schmadocs:
    i.metadata = {
        "Source": "NN_ADM_Data_Modeling",
        "ID": "Data-Model",
        "Creation_Date": "2026-07-05T18:55:24+05:30",
        "Author": "Naresh Neelam",
        "JIRA": "DM001"
    }

In [8]:
schmadocs

[Document(metadata={'Source': 'NN_ADM_Data_Modeling', 'ID': 'Data-Model', 'Creation_Date': '2026-07-05T18:55:24+05:30', 'Author': 'Naresh Neelam', 'JIRA': 'DM001'}, page_content='4.3 NN Sample Schema Table Descriptions \n \nConsider the columns of each table of HR sample schema. \n• Table NN.ADM_COUNTRIES \n• Table NN.ADM_DEPARTMENTS \n• Table NN.ADM_EMPLOYEES \n• Table NN.ADM_JOBS \n• Table NN.ADM_JOB_HISTORY \n• Table NN.ADM_LOCATIONS \n• Table NN.ADM_REGIONS \n \n43.1 Table NN.ADM_COUNTRIES \nTable NN.ADM_COUNTRIES Table Description \nColumn Name Null? Type \nCOUNTRY_ID NOT NULL CHAR(2) \nCOUNTRY_NAME   VARCHAR2(40) \nREGION_ID   NUMBER \n4.3.2 Table NN.ADM_DEPARTMENTS \nTable  NN.ADM_DEPARTMENTS Table Description \nColumn Name Null? Type \nDEPARTMENT_ID NOT NULL NUMBER(4) \nDEPARTMENT_NAME NOT NULL VARCHAR2(30) \nMANAGER_ID   NUMBER(6) \nLOCATION_ID   NUMBER(4) \n4.3.3 Table NN.ADM_EMPLOYEES \nTable  NN.ADM_EMPLOYEES Table Description \nColumn Name Null? Type \nEMPLOYEE_ID NOT NULL

### Splitting Documents into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

TexSplitter=RecursiveCharacterTextSplitter(
           chunk_size=1000,
           chunk_overlap=100
)

datachunks=TexSplitter.split_documents(schmadocs)


In [11]:
datachunks

[Document(metadata={'Source': 'NN_ADM_Data_Modeling', 'ID': 'Data-Model', 'Creation_Date': '2026-07-05T18:55:24+05:30', 'Author': 'Naresh Neelam', 'JIRA': 'DM001'}, page_content='4.3 NN Sample Schema Table Descriptions \n \nConsider the columns of each table of HR sample schema. \n• Table NN.ADM_COUNTRIES \n• Table NN.ADM_DEPARTMENTS \n• Table NN.ADM_EMPLOYEES \n• Table NN.ADM_JOBS \n• Table NN.ADM_JOB_HISTORY \n• Table NN.ADM_LOCATIONS \n• Table NN.ADM_REGIONS \n \n43.1 Table NN.ADM_COUNTRIES \nTable NN.ADM_COUNTRIES Table Description \nColumn Name Null? Type \nCOUNTRY_ID NOT NULL CHAR(2) \nCOUNTRY_NAME   VARCHAR2(40) \nREGION_ID   NUMBER \n4.3.2 Table NN.ADM_DEPARTMENTS \nTable  NN.ADM_DEPARTMENTS Table Description \nColumn Name Null? Type \nDEPARTMENT_ID NOT NULL NUMBER(4) \nDEPARTMENT_NAME NOT NULL VARCHAR2(30) \nMANAGER_ID   NUMBER(6) \nLOCATION_ID   NUMBER(4) \n4.3.3 Table NN.ADM_EMPLOYEES \nTable  NN.ADM_EMPLOYEES Table Description \nColumn Name Null? Type \nEMPLOYEE_ID NOT NULL

### Embeddings Model Preparation for DataChunks

In [12]:
from langchain_openai import OpenAIEmbeddings
embeddingModel=OpenAIEmbeddings(model="text-embedding-3-small")

### Created Embeddings and Store in Embedding in Vector DB

In [16]:
from langchain_community.vectorstores import Chroma

VectorDB=Chroma.from_documents(
    documents=datachunks,
    embedding=embeddingModel
)

### Semantic Search

In [17]:
VectorDB.similarity_search("*")

[Document(metadata={'Author': 'Naresh Neelam', 'Creation_Date': '2026-07-05T18:55:24+05:30', 'JIRA': 'DM001', 'Source': 'NN_ADM_Data_Modeling', 'ID': 'Data-Model'}, page_content='Column Name Null? Type \nJOB_ID NOT NULL VARCHAR2(10) \nSALARY   NUMBER(8,2) \nCOMMISSION_PCT   NUMBER(2,2) \nMANAGER_ID   NUMBER(6) \nDEPARTMENT_ID   NUMBER(4) \n4.3.4 Table NN.ADM_JOBS \nTable NN.ADM_JOBS Table Description \nColumn Name Null? Type \nJOB_ID NOT NULL VARCHAR2(10) \nJOB_TITLE NOT NULL VARCHAR2(35) \nMIN_SALARY   NUMBER(6) \nMAX_SALARY   NUMBER(6) \n4.3.5 Table NN.ADM_JOB_HISTORY \nTable NN.ADM_JOB_HISTORY Table Description \nColumn Name Null? Type \nEMPLOYEE_ID NOT NULL NUMBER(6) \nSTART_DATE NOT NULL DATE \nEND_DATE NOT NULL DATE \nJOB_ID NOT NULL VARCHAR2(10) \nDEPARTMENT_ID   NUMBER(4) \n4.3.6 Table NN.ADM_LOCATIONS \nTable NN.ADM_LOCATIONS Table Description \nColumn Name Null? Type \nLOCATION_ID NOT NULL NUMBER(4) \nSTREET_ADDRESS   VARCHAR2(40)'),
 Document(metadata={'Author': 'Naresh Neel

In [18]:
context=VectorDB.similarity_search("*")

In [30]:
from langchain_core.output_parsers import StrOutputParser
from pathlib import Path


response = llm.invoke(f"""Analyse all tables and columns and Create a Best Fit Facts and Dimensions - Data Modeling,
                         A columns level Mapping linease from source to Dim and Facts, and SnowFlake SQL Queries to populate dimensions and Facets
                         Output: Professional Raw HTML with CSS , Data Model With Data Model Diagrams using JavaScript (Mermaid.js)
                      for the: {context}""")

llmoutput=response.content

parser = StrOutputParser()
html_string = parser.parse(llmoutput)
Path("./Output/DataModel_NN_ADM.html").write_text(html_string, encoding="utf-8")



20200